# Track 1 — reproducible LFM radar simulation
Team Spaceship: Manas Vinod Jain and Vikhyat Gupta. This notebook executes the submitted Python engine. Link parameters are simulation assumptions, not verified equipment specifications.

In [ ]:
from pathlib import Path
import os, subprocess, sys
# Run from the repository root locally. Colab can fetch the submitted repository.
if not Path("track1_lfm_simulation.py").exists():
    repo = Path("space-radar-pulse-compression")
    if not repo.exists():
        subprocess.run(["git", "clone", "https://github.com/Manas-Jain-24/space-radar-pulse-compression.git", str(repo)], check=True)
    os.chdir(repo)
import track1_lfm_simulation as radar
import track1_bonus_upgrades as bonus
import matplotlib.pyplot as plt


## Setup and assumptions
PRT=2 ms, pulse widths=10/100 µs, bandwidth=4 MHz, range=200 km and RCS=1 m². The comparison uses equal peak power and receiver settings.

In [ ]:
print({"PRT_ms": radar.PRT*1e3, "pulse_widths_us": [radar.TAU_1*1e6, radar.TAU_2*1e6],
       "bandwidth_MHz": radar.B/1e6, "range_km": radar.R_TARGET/1e3, "RCS_m2": radar.RCS})
link = radar.compute_radar_link(radar.PT, radar.G_TX, radar.G_RX, radar.LAMBDA_C,
                               radar.RCS, radar.R_TARGET, radar.LOSS, radar.NF, radar.B, radar.FS)
print(link)


## Complete mandatory chain
Generates pulse train, waveforms/spectra, delayed noisy echoes, pulse compression and CA-CFAR plots. Saves the computed results JSON.

In [ ]:
results = radar.run_simulation(show=True)
plt.close("all")


## Quantitative results
Noisy peak/known noise variance is a single-realization statistic. Nominal resolution and full half-power width are different quantities.

In [ ]:
for key in ["snr_theory_1_db", "snr_theory_2_db", "snr_meas_1_db", "snr_meas_2_db", "delta_snr_meas_db",
            "mlw_1", "mlw_2", "margin_1_db", "margin_2_db", "cfar_margin_1", "cfar_margin_2"]:
    print(key, results[key])


## Production-chain Monte Carlo
The interval describes the mean enhancement, not probability of detection.

In [ ]:
print(results["monte_carlo"])


## Bonus waveform experiments
Two close targets (the default 50 m coherent pair has one central peak), Doppler coupling with up/down chirps, and receiver windowing.

In [ ]:
bonus.run_multitarget_experiment()
bonus.run_doppler_coupling_experiment()
bonus.run_windowing_experiment()
plt.close("all")
from IPython.display import Image, display
for name in ["fig6_multitarget_resolution.png", "fig7_doppler_coupling_and_chirp_diversity.png", "fig8_windowing_sidelobe_suppression.png"]:
    display(Image(filename=str(Path("figures/track1") / name)))


## Limitations
Fixed nonfluctuating target and AWGN. CA-CFAR nominal Pfa assumes independent exponential cells; oversampled matched-filter cells are correlated. A 100 µs pulse increases energy and average transmit power tenfold and has a 15 km transmit blind zone. No field reliability or zero-false-alarm guarantee is implied.